# 04 · Demand targets: does location predict commercial performance?

Features come from the **June 2026** snapshot (classmate's temporal panel); outcomes are observed afterwards (July reviews, September snapshot), so nothing from the future enters the features. Sets A, B, C, D as in notebook 02.

| Target | Type |
|---|---|
| July review count, with / without review history | Poisson regression (D² = share of Poisson deviance explained) |
| New listing (no review in June) gets a first review by September | classification (AUC) |
| Listing not visible in the September snapshot | classification (visibility, not a verified exit) |
| Non-superhost in June becomes superhost by September | classification |
| Review-derived nights and revenue over 12 months (price not a feature) | regression |

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
from config import CLASSMATE_REPO, SLIM, EXT, OUT
from common import *
from external import build
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
st = pd.read_csv(CLASSMATE_REPO / "data" / "stations_used.csv")   # official IDFM stations (classmate's repo)
print("classmate repo:", CLASSMATE_REPO.exists(), "| slim:", SLIM.exists(), "| external:", EXT.exists())

classmate repo: True | slim: True | external: True


In [2]:
tp = pd.read_csv(CLASSMATE_REPO / "data" / "insideairbnb" / "temporal_review_panel.csv.gz", low_memory=False)
J = tp[tp.split == "test"].reset_index(drop=True)                     # June 2026 source snapshot
X, G, info, _ = build(J.latitude.values, J.longitude.values, EXT, st); J = pd.concat([J, X], axis=1)
J["cell"] = cell_1km(J.latitude, J.longitude); J["block"] = block_3km(J.latitude, J.longitude); cat(J, ["room_type", "neighbourhood_cleansed"])
L = pd.read_csv(SLIM / "listings_slim.csv.gz", low_memory=False)
R = pd.read_csv(SLIM / "reviews_slim.csv.gz", usecols=["listing_id", "date"], parse_dates=["date"])
PROP = ["accommodates", "bedrooms", "beds", "bathrooms_n", "minimum_nights", "room_type", "capture_age_days"]
HIST = ["reviews_30", "reviews_90", "reviews_365", "reviews_total", "days_since_last_review", "days_since_first_review", "superhost", "review_scores_rating"]
EXT_ALL = sum(G.values(), []); GEO = ["latitude", "longitude", "neighbourhood_cleansed"]
def sets(A): return {"A internal": A, "B A+all external": A + EXT_ALL, "C A+coordinates": A + GEO, "D C+all external": A + GEO + EXT_ALL}
rows = []; print(len(J), "listings in the June snapshot")

77679 listings in the June snapshot


In [3]:
obs = J[J.followup_observed].reset_index(drop=True)
run_task("T2 July reviews (count)", obs, obs.target.values.astype(float), "poisson", sets(PROP + HIST), rows)
run_task("T2b July reviews, no history", obs, obs.target.values.astype(float), "poisson", sets(PROP), rows)
cold = obs[obs.reviews_total == 0].reset_index(drop=True)
r = R.merge(cold[["id", "last_scraped"]].assign(last_scraped=lambda x: pd.to_datetime(x.last_scraped)), left_on="listing_id", right_on="id")
y3 = (cold.id.map(r[r.date > r.last_scraped].groupby("listing_id").size()).fillna(0) > 0).astype(int).values
run_task("T3 cold start: first review", cold, y3, "clf", sets(PROP), rows)
run_task("T4 disappears by September", J, (~J.followup_observed).astype(int).values, "clf", sets(PROP + HIST), rows)
sh = L.set_index("id").host_is_superhost.eq("t")
c5 = obs[(obs.superhost == 0) & obs.id.isin(sh.index)].reset_index(drop=True)
run_task("T5 becomes superhost", c5, c5.id.map(sh).astype(int).values, "clf", sets(PROP + [h for h in HIST if h != "superhost"]), rows)

T2 July reviews (count)      host     A internal                         D2_poisson=0.6329  MAE=0.5235  Spearman=0.6755


T2 July reviews (count)      host     B A+all external                   D2_poisson=0.6302  MAE=0.5259  Spearman=0.6751


T2 July reviews (count)      host     C A+coordinates                    D2_poisson=0.6307  MAE=0.5252  Spearman=0.6751


T2 July reviews (count)      host     D C+all external                   D2_poisson=0.6297  MAE=0.5261  Spearman=0.6748


T2 July reviews (count)      spatial  A internal                         D2_poisson=0.6327  MAE=0.5229  Spearman=0.6761


T2 July reviews (count)      spatial  B A+all external                   D2_poisson=0.6294  MAE=0.5262  Spearman=0.6753


... (31 outputs left out)

T5 becomes superhost         spatial  B A+all external                   AUC=0.8603  PR_AUC=0.2820


T5 becomes superhost         spatial  C A+coordinates                    AUC=0.8607  PR_AUC=0.2851


T5 becomes superhost         spatial  D C+all external                   AUC=0.8602  PR_AUC=0.2807


{('host',
  'A internal'): array([0.07252467, 0.04669756, 0.37646624, ..., 0.02924537, 0.00912616,
        0.01390405], shape=(55069,)),
 ('host',
  'B A+all external'): array([0.06723437, 0.05133827, 0.25791531, ..., 0.03977655, 0.00702562,
        0.0069953 ], shape=(55069,)),
 ('host',
  'C A+coordinates'): array([0.05703991, 0.0827501 , 0.34929188, ..., 0.02597487, 0.0087103 ,
        0.00925825], shape=(55069,)),
 ('host',
  'D C+all external'): array([0.05309576, 0.04198357, 0.37285006, ..., 0.03990866, 0.0084699 ,
        0.0075117 ], shape=(55069,)),
 ('spatial',
  'A internal'): array([0.06811353, 0.02024072, 0.59103768, ..., 0.03191252, 0.00805511,
        0.00861975], shape=(55069,)),
 ('spatial',
  'B A+all external'): array([0.07645813, 0.02401666, 0.4771458 , ..., 0.0360124 , 0.00909756,
        0.00599332], shape=(55069,)),
 ('spatial',
  'C A+coordinates'): array([0.05078379, 0.02053756, 0.51984884, ..., 0.04061872, 0.00962999,
        0.00879563], shape=(55069,)),
 ('s

**Review-derived nights and revenue.** Inside Airbnb's rule: nights = reviews / 0.5 × max(5.2, minimum nights), capped at 70% of the year. Price is not a feature here, so the target is not partly mechanical.

In [4]:
pf = pd.read_csv(CLASSMATE_REPO / "data" / "price_features.csv.gz")
snap = pd.to_datetime(L.last_scraped).max(); cut = snap - pd.DateOffset(months=12)
pre, post = R[R.date < cut], R[(R.date >= cut) & (R.date < snap)]
pf["rev_pre"] = pf.id.map(pre.groupby("listing_id").size()).fillna(0)
pf["rev12_pre"] = pf.id.map(pre[pre.date >= cut - pd.DateOffset(months=12)].groupby("listing_id").size()).fillna(0)
pf["rev_post"] = pf.id.map(post.groupby("listing_id").size()).fillna(0)
w = pf[pf.rev_pre > 0].reset_index(drop=True)
nights = np.minimum(w.rev_post / 0.5 * np.maximum(5.2, w.minimum_nights.fillna(5.2).clip(upper=30)), 0.7 * 365)
Xw, _, _, _ = build(w.latitude.values, w.longitude.values, EXT, st); w = pd.concat([w, Xw], axis=1)
w["cell"] = cell_1km(w.latitude, w.longitude); w["block"] = block_3km(w.latitude, w.longitude); cat(w, ["property_type", "neighbourhood_cleansed"])
A6 = ["accommodates", "bedrooms", "beds", "bathrooms_n", "minimum_nights", "amenity_count", "wifi", "kitchen", "washer", "air_conditioning", "elevator", "property_type", "rev_pre", "rev12_pre"]
run_task("T6 log est. revenue (no price)", w, np.log1p(nights * w.price_eur).values, "reg", sets(A6), rows)
run_task("T6b log est. nights", w, np.log1p(nights).values, "reg", sets(A6), rows)

T6 log est. revenue (no price) host     A internal                         R2=0.4861  MAE=1.7430


T6 log est. revenue (no price) host     B A+all external                   R2=0.4832  MAE=1.7464


T6 log est. revenue (no price) host     C A+coordinates                    R2=0.4843  MAE=1.7393


T6 log est. revenue (no price) host     D C+all external                   R2=0.4817  MAE=1.7479


T6 log est. revenue (no price) spatial  A internal                         R2=0.4861  MAE=1.7774


T6 log est. revenue (no price) spatial  B A+all external                   R2=0.4822  MAE=1.7792


... (7 outputs left out)

T6b log est. nights          spatial  B A+all external                   R2=0.5356  MAE=0.9060


T6b log est. nights          spatial  C A+coordinates                    R2=0.5378  MAE=0.9000


T6b log est. nights          spatial  D C+all external                   R2=0.5344  MAE=0.9061


{('host',
  'A internal'): array([3.20797814, 4.46268983, 3.80701065, ..., 4.29715051, 4.80662357,
        5.11655333], shape=(21097,)),
 ('host',
  'B A+all external'): array([3.35648179, 4.77229742, 3.46798835, ..., 4.23511769, 4.82714003,
        5.05405614], shape=(21097,)),
 ('host',
  'C A+coordinates'): array([3.03644824, 4.89837496, 3.93304006, ..., 4.22494528, 4.76973888,
        5.06660541], shape=(21097,)),
 ('host',
  'D C+all external'): array([3.2254372 , 4.80603523, 3.74066475, ..., 4.21472706, 4.84497336,
        5.03755005], shape=(21097,)),
 ('spatial',
  'A internal'): array([3.21206651, 4.90391247, 3.65966899, ..., 4.23904294, 4.67165325,
        4.94397463], shape=(21097,)),
 ('spatial',
  'B A+all external'): array([3.40064274, 5.08406832, 3.52450436, ..., 4.11491641, 4.73567348,
        4.95169377], shape=(21097,)),
 ('spatial',
  'C A+coordinates'): array([3.0474848 , 4.81208089, 3.63652033, ..., 4.21634163, 4.62514925,
        4.94988118], shape=(21097,)),
 ('s

In [5]:
t2 = pd.DataFrame(rows); t2.to_csv(OUT / "t2_performance.csv", index=False)
t2["value"] = t2.apply(lambda r: r[r.primary], axis=1)
t2.pivot_table(index=["task", "n", "primary"], columns=["protocol", "features"], values="value", sort=False).round(3)

protocol                                              host                                                      spatial                                   \
features                                        A internal B A+all external C A+coordinates D C+all external A internal B A+all external C A+coordinates   
task                           n     primary                                                                                                               
T2 July reviews (count)        71588 D2_poisson      0.633            0.630           0.631            0.630      0.633            0.629           0.630   
T2b July reviews, no history   71588 D2_poisson      0.323            0.321           0.323            0.320      0.320            0.316           0.315   
T3 cold start: first review    14571 AUC             0.804            0.801           0.803            0.798      0.797            0.792           0.792   
T4 disappears by September     77679 AUC             0.699            0.695           0.697            0.694      0.696            0.688           0.692   
T5 becomes superhost           55069 AUC             0.858            0.854           0.854            0.854      0.862            0.860           0.861   
T6 log est. revenue (no price) 21097 R2              0.486            0.483           0.484            0.482      0.486            0.482           0.482   
T6b log est. nights            21097 R2              0.543            0.538           0.539            0.535      0.541            0.536           0.538   

protocol                                                          
features                                        D C+all external  
task                           n     primary                      
T2 July reviews (count)        71588 D2_poisson            0.629  
T2b July reviews, no history   71588 D2_poisson            0.314  
T3 cold start: first review    14571 AUC                   0.787  
T4 disappears by September     77679 AUC                   0.686  
T5 becomes superhost           55069 AUC                   0.860  
T6 log est. revenue (no price) 21097 R2                    0.479  
T6b log est. nights            21097 R2                    0.534

**Reading.** Neither external data nor coordinates improve any demand target; the small losses are noise from extra features. Coordinates also give nothing, so the reason is not the quality of external data. Removing review history (T2b) does not change this either. Location is priced in: better places cost more, which offsets their advantage.